# Semaine 10 — Jour 5 : Adaptation légère et contrôle des poids

Poursuivre réellement l’entraînement d’une tête linéaire sur représentation figée.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. L’installation de l’environnement ajoute un temps distinct; le lab LoRA est optionnel.

Prérequis : Python 3.11+, bases de classification et de vecteurs. Le parcours utilise NumPy et scikit-learn, sans modèle à télécharger ni clé API. Installer les dépendances S10 selon [le guide](../../../docs/week10/QUICKSTART.md).

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week10/S10_J5_fine_tuning_leger.ipynb` et `notebooks/week10/teacher/S10_J5_fine_tuning_leger_teacher.ipynb`.

Critère de réussite : Distinguer head-only adaptation, fine-tuning complet et LoRA.

Les apprentissages du parcours principal sont réellement exécutables. Le corpus reste fictif, les baselines à règles sont identifiées et aucun résultat LoRA n’est annoncé.


# Objectifs mesurables

- Poursuivre réellement l’entraînement d’une tête linéaire sur représentation figée.
- Sélectionner les epochs sur validation et constater les régressions ou absences de gain.
- Distinguer head-only adaptation, fine-tuning complet et LoRA.

## Preuves attendues

Une sortie reproductible, un invariant vérifié et une limite de la tâche correctement interprétée.


# S10 J5 — Adaptation légère et contrôle des poids


## 1. Adapter sans tout réentraîner
Une adaptation légère modifie une partie ciblée du système plutôt que tous ses paramètres. Le coût, les données nécessaires et les risques dépendent des paramètres choisis. Avant un grand modèle, le lab montre un cas entièrement exécuté : un vocabulaire/IDF TF-IDF figé et une tête linéaire déjà entraînée que l’on continue à optimiser.

Cette expérience est une adaptation de tête de classifieur, pas un fine-tuning de LLM ni une exécution LoRA. La distinguer évite de confondre le principe de gel des représentations avec les mécanismes d’un Transformer. Le dossier optionnel propose une vraie boucle LoRA SEQ_CLS sur modèle BERT local, non exécutée ici.

## 2. Ce qui reste fixe
Le vectoriseur est appris sur les 36 exemples train. Les 12 exemples adapt passent seulement par transform. Un terme nouveau absent du vocabulaire ne crée pas une nouvelle caractéristique. La tête peut modifier les poids de caractéristiques connues; elle ne peut pas apprendre une colonne qui n’existe pas.

adapt_head clone le modèle initial avant partial_fit. Le modèle baseline reste donc disponible pour la comparaison. Le code vérifie que la partition fournie est adapt et que le nombre d’epochs est un entier positif. Dans une expérience réelle, conserver en plus les artefacts, versions et paramètres d’optimisation.

## 3. Sélection contrôlée
Le lab essaie 1, 3 et 8 epochs, choisis avant de voir les résultats. Il sélectionne le meilleur macro-F1 de validation et départage une égalité en faveur du plus petit nombre d’epochs. Le seuil de décision est également choisi sur validation. Aucun texte test n’entre dans le fit ni la sélection.

Les trois variantes peuvent être à égalité sur ce corpus facile. Un résultat sans amélioration est une conclusion valide : l’adaptation a changé les poids, mais elle n’a pas démontré de bénéfice sur le test. Il ne faut ni fabriquer un gain ni choisir la meilleure valeur de test parmi plusieurs essais.

## 4. Régressions et oubli
Une adaptation sur un sous-domaine peut améliorer ses cas et dégrader l’ancien domaine. Comparer la baseline et le candidat sur un protocole stable, puis observer les résultats par catégorie. Une moyenne peut cacher une régression minoritaire. Avec peu d’exemples, la mesure est instable et doit être complétée avant une décision réelle.

Un mélange de rappel d’anciennes données, une régularisation ou une représentation différente peut aider selon la tâche. Le cours ne les applique pas automatiquement. Tout changement du vocabulaire exige de réentraîner une tête compatible plutôt que d’utiliser des poids d’une autre dimension.

## 5. LoRA : un autre mécanisme
LoRA ajoute des matrices de faible rang sur certaines couches d’un modèle préentraîné, en conservant les poids de base gelés. Le choix des modules dépend de l’architecture. Le script optionnel cible explicitement BERT, query/value et une tête classifier sauvegardée, plutôt que supposer les mêmes noms pour tout modèle.

Le script utilise des artefacts locaux, une sélection sur validation et n’exécute pas le test. PyTorch, Transformers, PEFT et le modèle ne sont pas disponibles dans la validation obligatoire. La boucle est fournie comme expérience à vérifier dans un environnement verrouillé, sans annoncer de score LoRA.

## 6. Persister sans confondre hash et confiance
Le module exporte vocabulaire, IDF, poids, classes, seuil et provenance dans un JSON sans pickle. La relecture contrôle formes, valeurs finies et empreinte de contenu. Cela permet un aller-retour d’inférence testable. Le hash détecte une modification par rapport à une référence fiable; il ne signe pas l’auteur et ne protège pas contre un remplacement coordonné du fichier et du hash.

## À retenir
Une adaptation est réussie si son protocole est correct et son intérêt mesuré, y compris quand le résultat conclut à une absence de gain.


## Diagramme du jour

```mermaid
flowchart TD
 A["Train : représentation et tête"] --> B["Baseline conservée"]
 B --> C["Clone de la tête"]
 D["Adapt : transform seulement"] --> C
 C --> E["Validation des epochs"]
 E --> F["Test après choix"]
 F --> G["Artefact et limites"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

NumPy et scikit-learn sont requis; aucune requête réseau dans les cellules.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week10/nlp.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week10.nlp import trained, artifact, classify, predict_artifact
bundle = trained()
base = artifact(bundle, "baseline")
candidate = artifact(bundle, "adapted")
assert base["vocabulary"] == candidate["vocabulary"]
assert base["idf"] == candidate["idf"]
assert base["coef"] != candidate["coef"]
print(bundle["selection"], "selected:", bundle["selected_epochs"])

# Lab J5 — Adaptation légère et contrôle des poids

Depuis la racine du dépôt, dans l’environnement S10 :

```bash
python3 book/week10/day05/labs/run.py
python3 -m examples.week10.demo --day 5
```

Résultat attendu : **vocabulaire inchangé, poids de tête modifiés; aucun gain de F1 imposé**.

## Travail

1. Exécuter le scénario et identifier la nature de chaque composant : appris, adapté ou à règles.
2. Réaliser l’exercice 2 sur une copie, sans modifier le corpus distribué.
3. Expliquer le dénominateur du résultat et une limite.

Les données sont sous `examples/week10/data`; le code commun sous `examples/week10/nlp.py`; les tests sous `tests/week10`. Lire [QUICKSTART](../../../../docs/week10/QUICKSTART.md) pour les dépendances, artefacts et commandes.

Les modèles entraînés sont conservés en mémoire dans chaque processus pour accélérer les notebooks. Une modification des données demande un nouveau processus ou de vider explicitement les caches trained/specialized avant de réentraîner. Les correctifs du notebook enseignant s’exécutent après le lab élève.


# Exercices

Conserver une première réponse avant les corrigés.

## Exercice 1 — Identifier les paramètres

Dans le lab, quels objets sont gelés et quels paramètres changent ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 2 — Gérer un mot nouveau

Un mot apparaît seulement dans adapt. Peut-il créer une colonne TF-IDF ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 3 — Choisir sans fuite

Trois essais ont le même F1 de validation. Comment départager sans consulter le test ?

Livrable : réponse argumentée avec calcul, sortie ou test.


# Entretien

Répondre en deux minutes par question avec un exemple et une limite.

## Question 1

Pourquoi ne pas appeler cela LoRA ?

## Question 2

Que livrer avec un adaptateur ?


# Challenge

Exporter baseline et candidat en JSON, vérifier que le vocabulaire est identique mais les poids différents, puis recharger l’inférence et comparer les probabilités à celles du modèle en mémoire.

## Critères

Protocole reproductible, invariant explicite et interprétation honnête du résultat, y compris sans amélioration.


# Références — Jour 5

Sources primaires consultées le 20 octobre 2026. Le corpus et les annotations du lab sont fictifs; les seuils sont pédagogiques.

- [scikit-learn 1.8 : pièges et fuite de données](https://scikit-learn.org/1.8/common_pitfalls.html).
- [SGDClassifier 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.SGDClassifier.html) — log_loss et partial_fit.
- [TfidfVectorizer 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Neighborhood Components Analysis 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html) — apprentissage métrique supervisé.
- [spaCy : entités nommées](https://spacy.io/usage/linguistic-features#named-entities) — vocabulaire et conventions; spaCy n’est pas utilisé dans le lab obligatoire.
- [PEFT : LoRA](https://huggingface.co/docs/peft/package_reference/lora) — extension Transformer optionnelle, non exécutée.

Lire en priorité les références du thème étudié. Les concepts communs à plusieurs bibliothèques ne rendent pas leurs objets ou artefacts interchangeables. Les dépendances obligatoires correspondent à l’environnement testé, les dépendances LoRA restent à verrouiller pour l’expérience optionnelle.
